# MuRIL BERT Scam Classifier — INT8 Quantization & Benchmark

This notebook performs **Dynamic INT8 Quantization** on the fine-tuned MuRIL ONNX binary scam classifier (`google/muril-base-cased`) and benchmarks both models.

### Objectives:
1. **Quantize** `scam_classifier.onnx` from FP32 (~238 MB) to true INT8 (~65 MB).
2. **Benchmark** latency (CPU execution matching mobile devices).
3. **Evaluate fidelity**: Check class prediction agreement and logit drift across Indian English and Hinglish scam samples.
4. **Download** the resulting `scam_classifier_int8.onnx` ready for drop-in replacement into Android `assets/muril/`.

## 1. Install Dependencies

In [ ]:
!pip install -q onnx onnxruntime transformers numpy matplotlib tabulate

## 2. Upload Model & Vocab Files
Upload your existing `scam_classifier.onnx` (or `scam_classifier_int8.onnx` if unquantized) and `vocab.txt`.

*Tip: You can also mount Google Drive or drag-and-drop the files into the left sidebar file explorer in Colab.*

In [ ]:
import os
from google.colab import files

# Check if model already exists (e.g. if uploaded via sidebar or Drive)
if not os.path.exists("scam_classifier.onnx"):
    print("Please upload 'scam_classifier.onnx' (or your ~238MB file):")
    uploaded = files.upload()
    # If uploaded under a different name (e.g., scam_classifier_int8.onnx), normalize name
    for fname in uploaded.keys():
        if fname.endswith(".onnx") and fname != "scam_classifier.onnx":
            os.rename(fname, "scam_classifier.onnx")
            print(f"Renamed {fname} -> scam_classifier.onnx")

if not os.path.exists("vocab.txt"):
    print("Please upload 'vocab.txt':")
    files.upload()

print(f"scam_classifier.onnx size: {os.path.getsize('scam_classifier.onnx') / (1024*1024):.2f} MB")
print(f"vocab.txt size: {os.path.getsize('vocab.txt') / (1024*1024):.2f} MB")

## 3. Dynamic INT8 Quantization
Quantizes the transformer linear and matrix multiplication weights (`MatMul`, `Gemm`) to 8-bit integers using per-channel scaling.

In [ ]:
import os
from onnxruntime.quantization import quantize_dynamic, QuantType

INPUT_MODEL = "scam_classifier.onnx"
OUTPUT_MODEL = "scam_classifier_int8_quantized.onnx"

print("Starting Dynamic INT8 Quantization...")
quantize_dynamic(
    model_input=INPUT_MODEL,
    model_output=OUTPUT_MODEL,
    weight_type=QuantType.QInt8,              # Signed 8-bit integers (standard for ARM / mobile CPUs)
    per_channel=True,                         # Highest accuracy across multi-head attention
    reduce_range=False,
    op_types_to_quantize=['MatMul', 'Gemm']   # Quantize heavy transformer weights
)

orig_size = os.path.getsize(INPUT_MODEL) / (1024 * 1024)
quant_size = os.path.getsize(OUTPUT_MODEL) / (1024 * 1024)
reduction = (1 - (quant_size / orig_size)) * 100

print("\n--- Quantization Complete ---")
print(f"Original Model Size : {orig_size:.2f} MB")
print(f"Quantized Model Size: {quant_size:.2f} MB")
print(f"Storage Savings     : {reduction:.1f}% reduction")

## 4. Setup Test Dataset & Tokenizer
We load MuRIL's WordPiece tokenizer and construct a representative test suite containing Indian English and Hinglish scam variations, along with benign transactional and social messages.

In [ ]:
import numpy as np
from transformers import BertTokenizer

# MuRIL is a cased model
tokenizer = BertTokenizer(vocab_file="vocab.txt", do_lower_case=False)

test_dataset = [
    # SCAM SAMPLES
    {"text": "Dear customer your SBI account KYC is expired. Account blocked. Update now: http://sbi-kyc-update.xyz", "expected": 1, "label": "KYC Phishing"},
    {"text": "Congratulations! You have won Rs 25,00,000 in KBC Kaun Banega Crorepati. Contact WhatsApp 9876543210", "expected": 1, "label": "Lottery Scam"},
    {"text": "Dear consumer, your electricity power will be disconnected at 9:30 PM tonight. Pay immediately: 9988776655", "expected": 1, "label": "Utility Scam"},
    {"text": "Part time job offer: Earn 5000 daily by liking YouTube videos and Google reviews. Telegram @hr_pay", "expected": 1, "label": "Task/Job Scam"},
    {"text": "Aapka loan approved ho gaya hai Rs 5,00,000 bina kisi document ke. Turant click kare: http://instantloan.in", "expected": 1, "label": "Hinglish Loan Scam"},
    {"text": "TRAI Warning: Your mobile number will be disconnected within 2 hours due to illegal activities. Dial 9 to connect", "expected": 1, "label": "SIM/TRAI Impersonation"},
    {"text": "Customs department notice: Parcel containing foreign currency detained in Mumbai airport. Pay clearance charge.", "expected": 1, "label": "Customs Scam"},
    
    # HAM (BENIGN) SAMPLES
    {"text": "Your OTP for HDFC Bank NetBanking transaction of INR 1,500.00 is 839201. Valid for 10 mins.", "expected": 0, "label": "Legit OTP"},
    {"text": "Your order #403-294829 has been shipped and will be delivered by tomorrow via BlueDart.", "expected": 0, "label": "Shipping Notification"},
    {"text": "Bhai, are we still meeting today at the library around 4 PM? Let me know.", "expected": 0, "label": "Personal Chat"},
    {"text": "Dear Customer, your electricity bill of Rs 1,420 for account 9021948 has been generated on bescom.org.", "expected": 0, "label": "Legit Utility Bill"},
    {"text": "Please find attached the slides for tomorrow's project presentation.", "expected": 0, "label": "Work Communication"},
    {"text": "Your salary of INR 65,000.00 credited to A/C XX4920 on 30-Sep-2026. Available balance INR 72,400.00.", "expected": 0, "label": "Bank Credit Alert"}
]

def tokenize_batch(texts, max_len=128):
    enc = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_tensors="np"
    )
    return {
        "input_ids": enc["input_ids"].astype(np.int64),
        "attention_mask": enc["attention_mask"].astype(np.int64),
        "token_type_ids": enc["token_type_ids"].astype(np.int64)
    }

texts = [item["text"] for item in test_dataset]
ground_truth = np.array([item["expected"] for item in test_dataset])
tokenized_inputs = tokenize_batch(texts)
print(f"Prepared {len(test_dataset)} test samples. Input tensor shape: {tokenized_inputs['input_ids'].shape}")

## 5. Benchmark Latency and Accuracy (Original vs. INT8)
We run repeated inferences configured with 2 CPU threads (mimicking mobile performance constraints).

In [ ]:
import time
import onnxruntime as ort
from tabulate import tabulate

# Configure sessions (2 threads to mimic mid-tier mobile cores)
opts = ort.SessionOptions()
opts.intra_op_num_threads = 2
opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_BASIC

session_orig = ort.InferenceSession(INPUT_MODEL, opts, providers=['CPUExecutionProvider'])
session_quant = ort.InferenceSession(OUTPUT_MODEL, opts, providers=['CPUExecutionProvider'])

num_samples = len(texts)

# Warm-up (prime instruction cache)
for _ in range(5):
    first_sample = {k: v[:1] for k, v in tokenized_inputs.items()}
    session_orig.run(None, first_sample)
    session_quant.run(None, first_sample)

# Measure Latencies
def measure_latency(sess, runs=30):
    times = []
    for _ in range(runs):
        for i in range(num_samples):
            sample = {k: v[i:i+1] for k, v in tokenized_inputs.items()}
            t0 = time.perf_counter()
            sess.run(None, sample)
            times.append((time.perf_counter() - t0) * 1000.0) # ms
    return np.mean(times), np.median(times), np.percentile(times, 95)

orig_mean, orig_p50, orig_p95 = measure_latency(session_orig)
quant_mean, quant_p50, quant_p95 = measure_latency(session_quant)

# Compute Predictions and Logits
def get_logits(sess):
    all_logits = []
    for i in range(num_samples):
        sample = {k: v[i:i+1] for k, v in tokenized_inputs.items()}
        logits = sess.run(None, sample)[0][0]
        all_logits.append(logits)
    return np.array(all_logits)

orig_logits = get_logits(session_orig)
quant_logits = get_logits(session_quant)

# Softmax calculation
def softmax(logits):
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / np.sum(exp, axis=-1, keepdims=True)

orig_probs = softmax(orig_logits)
quant_probs = softmax(quant_logits)

orig_preds = np.argmax(orig_probs, axis=1)
quant_preds = np.argmax(quant_probs, axis=1)

# Comparative Metrics
agreement = np.mean(orig_preds == quant_preds) * 100.0
mae_drift = np.mean(np.abs(orig_logits - quant_logits))
orig_acc = np.mean(orig_preds == ground_truth) * 100.0
quant_acc = np.mean(quant_preds == ground_truth) * 100.0

speedup = orig_p50 / quant_p50 if quant_p50 > 0 else 0

# Display Results Table
summary_data = [
    ["File Size (MB)", f"{orig_size:.2f} MB", f"{quant_size:.2f} MB", f"-{reduction:.1f}%"],
    ["Median Latency (ms)", f"{orig_p50:.2f} ms", f"{quant_p50:.2f} ms", f"{speedup:.2f}x faster"],
    ["95th % Latency (ms)", f"{orig_p95:.2f} ms", f"{quant_p95:.2f} ms", "-"],
    ["Test Set Accuracy", f"{orig_acc:.1f}%", f"{quant_acc:.1f}%", f"{quant_acc - orig_acc:+.1f}%"],
    ["Decision Agreement", "100.0%", f"{agreement:.1f}%", "Identical Decisions"],
    ["Mean Logit Drift", "0.000", f"{mae_drift:.4f}", "Minimal Error"]
]

print("\n" + tabulate(summary_data, headers=["Metric", "Original (FP32)", "Quantized (INT8)", "Delta / Gain"], tablefmt="fancy_grid"))

## 6. Sample-by-Sample Prediction Comparison
Inspect probabilities for each individual message to ensure borderline cases remain robust.

In [ ]:
rows = []
for i, item in enumerate(test_dataset):
    short_text = (item['text'][:45] + '...') if len(item['text']) > 45 else item['text']
    orig_label = "SCAM" if orig_preds[i] == 1 else "HAM"
    orig_conf = orig_probs[i][orig_preds[i]] * 100
    quant_label = "SCAM" if quant_preds[i] == 1 else "HAM"
    quant_conf = quant_probs[i][quant_preds[i]] * 100
    match = "✅" if orig_preds[i] == quant_preds[i] else "❌"
    rows.append([i+1, short_text, f"{orig_label} ({orig_conf:.1f}%)", f"{quant_label} ({quant_conf:.1f}%)", match])

print(tabulate(rows, headers=["#", "Message Snippet", "Original Output", "INT8 Output", "Match"], tablefmt="grid"))

## 7. Visualizations (Latency & Confidence Drift)

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Size and Latency Comparison
labels = ['Original (FP32)', 'Quantized (INT8)']
sizes = [orig_size, quant_size]
latencies = [orig_p50, quant_p50]

color_orig = '#e74c3c'
color_quant = '#2ecc71'

ax1.bar([0, 1], sizes, color=[color_orig, color_quant], width=0.4, align='center', alpha=0.85)
ax1.set_xticks([0, 1])
ax1.set_xticklabels(labels, fontweight='bold')
ax1.set_ylabel('Model Size (MB)', fontweight='bold')
ax1.set_title('Disk Storage Footprint', fontweight='bold', fontsize=12)
for i, v in enumerate(sizes):
    ax1.text(i, v + 5, f"{v:.1f} MB", ha='center', fontweight='bold')

# Plot 2: Latency
ax2.bar([0, 1], latencies, color=[color_orig, color_quant], width=0.4, align='center', alpha=0.85)
ax2.set_xticks([0, 1])
ax2.set_xticklabels(labels, fontweight='bold')
ax2.set_ylabel('Median Latency (ms)', fontweight='bold')
ax2.set_title('Inference Latency (2 CPU Threads)', fontweight='bold', fontsize=12)
for i, v in enumerate(latencies):
    ax2.text(i, v + 2, f"{v:.1f} ms", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Download the Quantized Model for Android
This downloads `scam_classifier_int8.onnx` directly to your local machine. Drop this file into `c:\FYP\app\src\main\assets\muril\scam_classifier_int8.onnx`.

In [ ]:
from google.colab import files
import shutil

# Rename to match Android expected asset filename
TARGET_FILE = "scam_classifier_int8.onnx"
shutil.copy(OUTPUT_MODEL, TARGET_FILE)

print(f"Downloading {TARGET_FILE} ({os.path.getsize(TARGET_FILE) / (1024*1024):.2f} MB)...")
files.download(TARGET_FILE)